<a href="https://colab.research.google.com/github/redinbluesky/handson-llm/blob/main/04_텍스트_분류.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 4 서론](#chapter4)
    * [Chapter 4-1 영화 리뷰 데이터셋](#chapter4-1)
    * [Chapter 4-2 표현 모델로 텍스트 분류하기](#chapter4-2)

## Chapter 4 서론 <a class="anchor" id="chapter4"></a>
1. 가장 일반적인 자연어 처리 작업은 분류이다.
    - 작업의 목표는 입력 텍스트에 레이블 혹은 클래스를 할당하는 것이 목표이다.
    - 예를 들어, 스팸 이메일 분류, 감정 분석, 주제 분류 등이 있다.
    
        ![분류 예시](./image/04_example.png)

2. 이 장에서 언어 모델을 텍스트 분류에 사용하는 몇 가지 방법을 알아본다.
    - 사전 훈련된 모델을 사용하는 방법을 소개한다.
    - 작업에 특화된 모델과 임베딩 모델을 다루어본다.

3. 대규모 데이터에서 이미 훈련되어 텍스트 분류에 활용할 수 있는 사전 훈련된 언어모델을 활용하는데 초점을 맞춘다.

4. 표현 언어 모델과 생성 언어 모델을 살펴보고 둘의 차이점을 알아본다.

    ![언어 모델 비교](./image/04_language_model_comparison.png)

## Chapter 4-1 영화 리뷰 데이터셋 <a class="anchor" id="chapter4-1"></a>
1. 허깅 페이스 허브에는 모델은 물론 텍스트 분류을 위한 데이터도 찾을 수 있다.
    - 유명한 로튼 토마토 데이터셋을 사용하여 모델을 훈련하고 평가한다.
    - 로트 토마토에서 수집한 양성 영화리뷰 5,331개, 음성 영화 리뷰 5,331개가 들어있다.

In [ ]:
# datasets 패키지를 사용하여 로튼 토마토 데이터 셋을 로드한다.
from datasets import load_dataset
data = load_dataset("rotten_tomatoes")
data

README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

2. 데이터는 훈련, 검증, 테스트 세트로 나누어져 있다.
    - 모델 훈련을 위해 훈련 세트를 사용하고, 결과 확인을 위해 테스트 세트를 사용한다.
    - 모델 훈련 시 하이퍼파라미터를 튜닝하고 싶다면 검증 세트를 사용할 수 있다.

In [10]:
# 훈련 세트 샘플 확인
data["train"][0]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

In [ ]:
# 훈련 세트 샘플 확인
#   - 첫 번째 샘풀과 마자막 샘플을 확인한다.
data["train"][0,-1]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 0]}

3. 리뷰에는 양성(0)과 음성(1) 레이블이 부여된 것으로보아 이진 분류 문제임을 알 수 있다

## Chapter 4-2 표현 모델로 텍스트 분류하기 <a class="anchor" id="chapter4-2"></a>
1. 사전 훈련된 표현 모델을 사용해 분류를 할때 일반적으로 작업에 특화된 모델이나 임베딩 모델을 사용한다.

2. 이런 모델은 아래의 그림과 같이 BERT와 같은 파운데이션 모델을 특정 후속 작업에 맞춰 미세 튜닝하여 만든다.

    ![표현 모델](./image/04_representation_model.png)

3. 이 장에서는 아래의 그림과 같이 두 모델을 동결하고(훈련하지 않고) 모델의 출력만 사용한다.
    - 이미 다른 사람이 미세 튜닝한 사전 훈련된 모델을 사용해 영화 리뷰를 분류한다.

    ![표현 모델 활용](./image/04_representation_model_usage.png)